# Lesson 03: Transformer Block

Goal: connect attention to the full block used repeatedly in Transformer models.

## Learning goals

- Identify the main components of a Transformer block
- Understand residual connections and layer normalization
- See where the feed-forward network (MLP) fits
- Run a tiny NumPy forward pass

## Decoder-style block (simplified)

A common pre-norm block looks like:
1. `x1 = x + Attention(LN(x))`
2. `x2 = x1 + MLP(LN(x1))`

Residual paths help optimization and gradient flow across deep stacks.

In [ ]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)

def softmax(x, axis=-1):
    x_shift = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x_shift)
    return e / np.sum(e, axis=axis, keepdims=True)

def layer_norm(x, eps=1e-5):
    mean = x.mean(axis=-1, keepdims=True)
    var = ((x - mean) ** 2).mean(axis=-1, keepdims=True)
    return (x - mean) / np.sqrt(var + eps)

In [ ]:
# Tiny setup
rng = np.random.default_rng(0)
seq_len = 4
d_model = 8
d_k = 4
d_ff = 16

x = rng.normal(size=(seq_len, d_model))

W_Q = rng.normal(scale=0.2, size=(d_model, d_k))
W_K = rng.normal(scale=0.2, size=(d_model, d_k))
W_V = rng.normal(scale=0.2, size=(d_model, d_k))
W_O = rng.normal(scale=0.2, size=(d_k, d_model))

W1 = rng.normal(scale=0.2, size=(d_model, d_ff))
b1 = np.zeros((d_ff,))
W2 = rng.normal(scale=0.2, size=(d_ff, d_model))
b2 = np.zeros((d_model,))

In [ ]:
# Attention sublayer
x_ln = layer_norm(x)
Q = x_ln @ W_Q
K = x_ln @ W_K
V = x_ln @ W_V

scores = (Q @ K.T) / np.sqrt(d_k)
weights = softmax(scores, axis=-1)
attn = weights @ V
attn_proj = attn @ W_O

x1 = x + attn_proj

print('x shape:', x.shape)
print('attn_proj shape:', attn_proj.shape)
print('x1 shape:', x1.shape)

In [ ]:
# MLP sublayer
x1_ln = layer_norm(x1)
hidden = np.maximum(0, x1_ln @ W1 + b1)  # ReLU
mlp_out = hidden @ W2 + b2
x2 = x1 + mlp_out

print('hidden shape:', hidden.shape)
print('mlp_out shape:', mlp_out.shape)
print('block output shape:', x2.shape)

## Why stacking works

Each block refines token representations:
- Attention mixes information across positions
- MLP transforms each position nonlinearly
- Residuals preserve useful signal while allowing new features

## Checkpoints

1. Why do we use residual connections around both sublayers?
2. What is the role of the MLP if attention already mixes tokens?
3. In one sentence, what does layer norm stabilize?

## Next lesson

`04_pretraining_objectives.ipynb`: what objective we train with (next-token, masked LM, and variants).